## Module 6-2 Classification with XGBoost: Predicting the Direction of Earnings Changes

This class turns a classic AccFin prediction problem into a hands-on machine learning exercise, following:

> Chen, X., Y. H. (T.) Cho, Y. Dou, and B. Lev (2022). Predicting future earnings changes using machine learning and detailed financial data. *Journal of Accounting Research* 60(2): 467-515.

Chen et al. (2022) use two tree-ensemble methods (random forests and stochastic gradient boosting) with over 13,000 XBRL-based predictors to forecast the *direction* of one-year-ahead earnings changes, and show they substantially outperform a "kitchen sink" logistic regression benchmark (AUC of ~67-69% vs. ~62%).

We replicate the same idea at class scale:

- **Data**: `data/comp_sample.csv` (Compustat annual fundamentals), instead of the paper's 13,881 XBRL tags.
- **Model**: `xgboost`, a modern, fast, and widely-used gradient boosting library, instead of the paper's random forest / stochastic gradient boosting implementations.
- **Target**: the same *drift-adjusted* direction of next-year EPS change used in the paper.

Everything else about the workflow — building a wide set of financial-statement predictors without hand-picking them, splitting chronologically instead of randomly, benchmarking against logistic regression, and inspecting feature importance — mirrors the paper's methodology.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

import xgboost as xgb

pd.set_option('display.max_columns', 50)

## 1. Load and filter the data

We reuse `data/comp_sample.csv`, the same Compustat annual-fundamentals extract used in Module 2's Sloan (1996) replication. As in that module, we keep industrial firms reporting in USD with active (`costat == 'A'`) status.

*Note: this notebook auto-detects whatever numeric financial-statement columns are present in the file (Section 3). If you later swap in a richer extract with more Compustat fields, the feature-engineering pipeline below will pick them up automatically — no code changes needed.*

In [ ]:
df = pd.read_csv('data/comp_sample.csv')

df = df[
    (df['indfmt'] == 'INDL') &
    (df['curcd'] == 'USD') &
    (df['costat'] == 'A')
].copy()

df = df.dropna(subset=['gvkey', 'fyear']).copy()
df['fyear'] = df['fyear'].astype(int)
df = df.sort_values(['gvkey', 'fyear']).reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
# A quick look at data quality: how much of each column is missing?
# XGBoost can handle this natively; a logistic regression benchmark cannot (Section 5).
df.isna().mean().sort_values(ascending=False)

## 2. Build the drift-adjusted earnings-change label

Following Chen et al. (2022, Section 3.2.1), we predict the **direction of next-year EPS change, net of the firm's own trend**. Three steps:

1. Compute EPS = `ni / csho`, and its year-over-year change, $\Delta EPS_t = EPS_t - EPS_{t-1}$.
2. Compute each firm's **drift**: the average $\Delta EPS$ over its trailing years, known as of year $t$. The paper uses a trailing 4-year window; because our sample only spans 2014-2023, we use *however many prior years are available* (minimum 1) so firms aren't dropped just for having a short history.
3. Label year $t$ as an **earnings increase (1)** if next year's change, net of drift, is positive: $\Delta EPS_{t+1} - drift_t > 0$, else **0**.

De-trending this way mitigates class imbalance (raw earnings increases outnumber decreases) and makes the label economically meaningful: we're predicting a *surprise* relative to the firm's own trajectory, not just "earnings went up."

In [ ]:
# EPS and its year-over-year change, computed within each firm (gvkey)
df['eps'] = (df['ni'] / df['csho']).replace([np.inf, -np.inf], np.nan)
df['d_eps'] = df.groupby('gvkey')['eps'].diff()

# Drift: trailing (up to) 4-year average change in EPS, known as of year t
df['drift'] = (
    df.groupby('gvkey')['d_eps']
      .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
)

# Next year's change in EPS, and the drift-adjusted version used for the label
df['d_eps_lead'] = df.groupby('gvkey')['d_eps'].shift(-1)
df['adj_d_eps_lead'] = df['d_eps_lead'] - df['drift']

df['label'] = np.where(
    df['adj_d_eps_lead'].isna(), np.nan, (df['adj_d_eps_lead'] > 0).astype(float)
)

df[['gvkey', 'fyear', 'eps', 'd_eps', 'drift', 'd_eps_lead', 'adj_d_eps_lead', 'label']].head(10)

In [ ]:
# How many firm-years have a usable label, and how balanced are the two classes?
print(df['label'].notna().sum(), 'firm-years with a usable label')
df['label'].value_counts(normalize=True)

## 3. Feature engineering: current value, lagged value, and % change

A key idea in Chen et al. (2022) is to throw in *every available financial-statement item* — current value, lagged value, and percentage change — rather than hand-picking a handful of ratios, and let the tree ensemble find the useful nonlinearities and interactions on its own.

We do the same thing generically: instead of hard-coding column names, we **auto-detect every numeric column** in the data (after excluding identifiers, auditor codes, and the columns we just used to build the label). That way, if you swap in a wider Compustat extract later, the predictor set — and every cell below it — grows automatically.

Dollar-denominated items are scaled by total assets (current value by $Assets_t$, lagged value by $Assets_{t-1}$), matching the paper. Total assets itself, shares outstanding, and share price are per-share/scale items and are kept unscaled, also matching the paper's treatment of "total assets itself and items on a per-share basis."

In [ ]:
# Columns to exclude from auto-detection: identifiers, non-financial codes,
# and everything we derived while building the label (Section 2).
ID_COLS = ['gvkey', 'datadate', 'fyear']
LABEL_COLS = ['eps', 'd_eps', 'drift', 'd_eps_lead', 'adj_d_eps_lead', 'label']
NON_FS_COLS = ['au']  # e.g. auditor code: an identifier, not a financial-statement item

# Items that should NOT be scaled by total assets: the denominator itself,
# plus per-share / share-count items.
NO_SCALE_COLS = ['at', 'csho', 'prcc_f']

exclude_cols = set(ID_COLS + LABEL_COLS + NON_FS_COLS)
numeric_cols = df.select_dtypes(include='number').columns
predictor_base_cols = [c for c in numeric_cols if c not in exclude_cols]

print(f'{len(predictor_base_cols)} raw financial-statement columns auto-detected as predictors:')
print(predictor_base_cols)

In [ ]:
def build_features(data: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """Current value, lagged value, and percentage change for each base column."""
    at_cur = data['at']
    at_lag = data.groupby('gvkey')['at'].shift(1)

    feature_cols = {}
    for col in base_cols:
        cur = data[col]
        lag = data.groupby('gvkey')[col].shift(1)

        if col in NO_SCALE_COLS:
            cur_scaled, lag_scaled = cur, lag
        else:
            cur_scaled = (cur / at_cur).replace([np.inf, -np.inf], np.nan)
            lag_scaled = (lag / at_lag).replace([np.inf, -np.inf], np.nan)

        pct_change = ((cur - lag) / lag.abs()).replace([np.inf, -np.inf], np.nan)

        feature_cols[f'{col}_cur'] = cur_scaled
        feature_cols[f'{col}_lag'] = lag_scaled
        feature_cols[f'{col}_pctchg'] = pct_change

    return pd.DataFrame(feature_cols, index=data.index)


X_all = build_features(df, predictor_base_cols)
print(X_all.shape)
X_all.head()

In [ ]:
# Assemble the modeling frame: identifiers + label + features.
# We only keep rows with a usable label; missing feature values are left as NaN
# on purpose (Section 5 shows why that matters).
feature_cols = X_all.columns.tolist()

model_df = pd.concat([df[['gvkey', 'fyear', 'label']], X_all], axis=1)
model_df = model_df.dropna(subset=['label']).reset_index(drop=True)

print(model_df.shape)
model_df.groupby('fyear').size()

## 4. Split chronologically, not randomly

A random `train_test_split` (or k-fold CV) would let the model "see the future": a 2022 observation could end up in the training set while its own firm's 2020 observation is used for testing. Chen et al. (2022, Section 3.2.3) make exactly this point and use a **rolling, chronological** split instead — train on the earliest years, validate on the next year, and test on the most recent years.

We use a simplified, single chronological split (rather than the paper's year-by-year rolling window) to keep the class focused on the modeling workflow:

- **Train**: earlier fiscal years
- **Validation**: used for XGBoost's early stopping (Section 6)
- **Test**: the most recent fiscal years, held out for the final comparison in Section 7

In [ ]:
TRAIN_END = 2019   # fiscal years <= this go to training
VAL_END = 2021     # fiscal years in (TRAIN_END, VAL_END] go to validation
                    # everything after VAL_END goes to the test set

train = model_df[model_df['fyear'] <= TRAIN_END]
val = model_df[(model_df['fyear'] > TRAIN_END) & (model_df['fyear'] <= VAL_END)]
test = model_df[model_df['fyear'] > VAL_END]

X_train, y_train = train[feature_cols], train['label']
X_val, y_val = val[feature_cols], val['label']
X_test, y_test = test[feature_cols], test['label']

print(f'train: {X_train.shape} ({train.fyear.min()}-{train.fyear.max()})')
print(f'val:   {X_val.shape} ({val.fyear.min()}-{val.fyear.max()})')
print(f'test:  {X_test.shape} ({test.fyear.min()}-{test.fyear.max()})')

## 5. Benchmark: a "kitchen sink" logistic regression

Before reaching for a more advanced model, we fit a plain logistic regression on the **same** features — this is the paper's Ou and Penman (1989)-style benchmark. `LogisticRegression` cannot handle missing values, so we first median-impute; it also converges faster and more reliably once features are on comparable scales, so we standardize after imputing. Keep both steps in mind: they are exactly the kind of preprocessing that tree ensembles like XGBoost are designed to avoid needing.

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)   # fit the imputer on training data only
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_imp = scaler.fit_transform(X_train_imp)   # fit the scaler on training data only
X_val_imp = scaler.transform(X_val_imp)
X_test_imp = scaler.transform(X_test_imp)

logit = LogisticRegression(max_iter=1000)
logit.fit(X_train_imp, y_train)

logit_test_auc = roc_auc_score(y_test, logit.predict_proba(X_test_imp)[:, 1])
print(f'Logistic regression benchmark - test AUC: {logit_test_auc:.3f}')

## 6. XGBoost

[`xgboost`](https://xgboost.readthedocs.io/) is a fast, regularized implementation of gradient-boosted trees — the same family of model as the stochastic gradient boosting Chen et al. (2022) use, but faster, better-regularized against overfitting, and (critically for us) able to split on missing values natively.

### 6.1 A baseline model

We fit `XGBClassifier` directly on `X_train`, **NaNs and all** — no imputation needed.

In [ ]:
xgb_baseline = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    importance_type='gain',
    eval_metric='auc',
    random_state=42,
)
xgb_baseline.fit(X_train, y_train)

baseline_test_auc = roc_auc_score(y_test, xgb_baseline.predict_proba(X_test)[:, 1])
print(f'XGBoost (untuned baseline) - test AUC: {baseline_test_auc:.3f}')

### 6.2 Tuning with early stopping

Rather than a full grid search, we let XGBoost grow up to 2,000 trees but **stop as soon as validation AUC stops improving** for 50 rounds in a row. This reuses the same train/validation split from Section 4 — echoing how the paper selects its model configuration on a held-out validation year before touching the test set.

In [ ]:
xgb_tuned = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    importance_type='gain',
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=42,
)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f'Stopped after {xgb_tuned.best_iteration + 1} trees '
      f'(validation AUC = {xgb_tuned.best_score:.3f})')

tuned_test_auc = roc_auc_score(y_test, xgb_tuned.predict_proba(X_test)[:, 1])
print(f'XGBoost (early-stopped) - test AUC: {tuned_test_auc:.3f}')

## 7. Evaluate and compare

We use **ROC-AUC** — the same metric as the paper — because it doesn't depend on picking a probability threshold, and it's directly comparable to the paper's reported numbers (Ou-Penman logit ~ 61.8% vs. their XGBoost-family models ~ 67-69%).

In [ ]:
plt.figure(figsize=(6, 6))

models_to_plot = {
    'Logistic regression': logit.predict_proba(X_test_imp)[:, 1],
    'XGBoost (baseline)': xgb_baseline.predict_proba(X_test)[:, 1],
    'XGBoost (early-stopped)': xgb_tuned.predict_proba(X_test)[:, 1],
}

for name, proba in models_to_plot.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random guess (AUC = 0.500)')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('Predicting the direction of next-year earnings changes')
plt.legend()
plt.show()

In [ ]:
# A confusion matrix and classification report at the conventional 0.5 cutoff
y_pred = (xgb_tuned.predict_proba(X_test)[:, 1] > 0.5).astype(int)

print('Confusion matrix (rows = actual, columns = predicted):')
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['Decrease', 'Increase']))

## 8. Which predictors matter most?

XGBoost's built-in `feature_importances_` reports each feature's average **gain**: how much it improved the model's loss function, summed across every split that uses it. It's a fast, model-level summary — useful for a first look, but it can't tell us *how* a feature pushes an individual firm-year's prediction up or down, or which direction the relationship runs.

That's exactly the gap the optional follow-on class on Explainable AI (`shap`) fills, following Parker, Jiang, Cho, and Vasarhelyi (2025, *The Accounting Review*).

In [ ]:
importances = pd.Series(
    xgb_tuned.feature_importances_, index=feature_cols
).sort_values(ascending=False)

top_n = 20
plt.figure(figsize=(8, 6))
importances.head(top_n).sort_values().plot(kind='barh')
plt.xlabel('XGBoost feature importance (gain)')
plt.title(f'Top {top_n} predictors of earnings-change direction')
plt.tight_layout()
plt.show()

**What did we simplify, relative to the paper?**

- **Predictors**: a few dozen Compustat items (current, lag, % change) instead of 13,881 XBRL-tagged items, including detailed footnote disclosures.
- **Model selection**: one chronological train/validation/test split with early stopping, instead of a year-by-year rolling window with a hyperparameter grid search.
- **Evaluation**: classification metrics only (ROC-AUC, confusion matrix). We stop short of the paper's economic-significance analysis, which forms long/short hedge portfolios from the predicted probabilities and computes size-adjusted stock returns — a natural extension if you have CRSP return data handy.